# Notebook Final — KNN

Alur:
1. Baca `data/train.csv` dan `data/test.csv` (`dtype={"id": str}`).
2. Split validasi baku kelompok (seed 42, ~20% per kelas).
3. Ringkas hasil eksperimen Fase 1–3 (`output/knn_experiment_results.csv`, `output/knn_decisions.csv`, `output/knn_feature_engineering_decisions.csv`) sebagai dasar pemilihan konfigurasi.
4. Evaluasi konfigurasi terpilih: **KNN k=21, weights=distance, metric=manhattan, cat_weight=1.0, threshold=0.5** (macro F1 validasi = 0.841689).
5. Latih ulang prapemrosesan + model pada **seluruh** `train.csv`.
6. Prediksi `test.csv` → `output/submission_knn.csv`.
7. Simpan model + prapemroses ke `output/final/model/knn/`.
8. Muat ulang dari disk pada sesi bersih dan pastikan prediksi identik.

Catatan eksperimen (seed 42, split baku, macro F1):
- Grid k (13–41): flat, terbaik k=21 → TOLAK perubahan.
- Threshold 0.30–0.70: flat di 0.50 → TOLAK.
- Metric: **manhattan 0.841689 vs euclidean 0.831783 (+0.0099) → ADOPSI**.
- cat_weight {0.5, sqrt(10), 2}: terbaik 0.5 = 0.843450 tapi delta +0.0018 < 0.005 → TOLAK, tetap 1.0.
- Feature engineering (total_guests, total_nights, total_stay_value, requests_per_guest): seluruhnya menurunkan macro F1 → TOLAK semua, fitur baseline dipertahankan.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (confusion_matrix, f1_score,
                             precision_recall_fscore_support)

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "data" / "train.csv").exists(), "Jalankan notebook dari akar repo"
sys.path.insert(0, str(REPO_ROOT / "src" / "experiments" / "knn"))

from knn import KNNClassifier
from preprocessing import KNNPreprocessor

SEED = 42
FINAL = {"k": 21, "weights": "distance", "metric": "manhattan",
         "cat_weight": 1.0, "threshold": 0.5}
BATCH = 2000
MODEL_DIR = REPO_ROOT / "output" / "final" / "model" / "knn"
print("numpy", np.__version__, "| pandas", pd.__version__)

## 1. Data dan split validasi baku kelompok

In [ ]:
train = pd.read_csv(REPO_ROOT / "data" / "train.csv", dtype={"id": str})
test = pd.read_csv(REPO_ROOT / "data" / "test.csv", dtype={"id": str})
y = train["is_canceled"].to_numpy()
assert set(np.unique(y)) == {0, 1}
X_raw = train.drop(columns=["id", "is_canceled"])

rng = np.random.default_rng(SEED)
valid_idx = np.sort(np.concatenate([
    rng.permutation(np.flatnonzero(y == label))[:int(np.ceil(0.2 * np.sum(y == label)))]
    for label in (0, 1)
]))
train_idx = np.setdiff1d(np.arange(len(y)), valid_idx)
print(f"train={len(train_idx)} baris, valid={len(valid_idx)} baris")
print(f"distribusi train: {np.bincount(y[train_idx])}, valid: {np.bincount(y[valid_idx])}")

## 2. Evaluasi konfigurasi final pada validasi

In [ ]:
pre = KNNPreprocessor(cat_weight=FINAL["cat_weight"]).fit(X_raw.iloc[train_idx])
X_tr = pre.transform(X_raw.iloc[train_idx])
X_va = pre.transform(X_raw.iloc[valid_idx])

model = KNNClassifier(n_neighbors=FINAL["k"], weights=FINAL["weights"],
                      metric=FINAL["metric"]).fit(X_tr, y[train_idx])
scores = model.decision_scores(X_va, batch_size=BATCH)
pred = (scores > FINAL["threshold"]).astype(np.int64)

macro = f1_score(y[valid_idx], pred, labels=[0, 1], average="macro", zero_division=0)
cm = confusion_matrix(y[valid_idx], pred, labels=[0, 1])
p, r, f, _ = precision_recall_fscore_support(y[valid_idx], pred, labels=[0, 1], zero_division=0)
print(f"macro F1 validasi = {macro:.6f}")
print(f"confusion matrix [0,1]:\n{cm}")
for i, lab in enumerate((0, 1)):
    print(f"kelas {lab}: precision={p[i]:.4f} recall={r[i]:.4f} f1={f[i]:.4f}")

Konfigurasi dari nol ini dibandingkan dengan `sklearn.neighbors.KNeighborsClassifier` pada metric yang sama di notebook eksperimen; selisih prediksi wajar hanya pada tie-breaking. Nilai 0.841689 konsisten dengan hasil `eksperimen_knn.py` Fase 1.

## 3. Latih ulang pada seluruh train dan prediksi test

In [ ]:
pre_full = KNNPreprocessor(cat_weight=FINAL["cat_weight"]).fit(X_raw)
X_full = pre_full.transform(X_raw)
X_test = pre_full.transform(test.drop(columns=["id"]))

model_full = KNNClassifier(n_neighbors=FINAL["k"], weights=FINAL["weights"],
                           metric=FINAL["metric"]).fit(X_full, y)
test_scores = model_full.decision_scores(X_test, batch_size=BATCH)
test_pred = (test_scores > FINAL["threshold"]).astype(int)

submission = pd.DataFrame({"id": test["id"], "is_canceled": test_pred})
OUT = REPO_ROOT / "output" / "submission_knn.csv"
submission.to_csv(OUT, index=False)

assert list(submission.columns) == ["id", "is_canceled"]
assert len(submission) == len(test)
assert submission["id"].tolist() == test["id"].tolist()
assert set(submission["is_canceled"].unique()).issubset({0, 1})
print(f"submission_knn.csv: {len(submission)} baris, distribusi {submission['is_canceled'].value_counts().to_dict()}")

## 4. Simpan model + prapemroses, muat ulang, cek identik

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
np.savez_compressed(MODEL_DIR / "knn_model.npz",
                    X_train=model_full.X_train, y_train=model_full.y_train)
(MODEL_DIR / "preprocessing.json").write_text(json.dumps(pre_full.to_dict(), ensure_ascii=False, indent=2))
(MODEL_DIR / "konfigurasi.json").write_text(json.dumps(
    {"algorithm": "KNN from scratch", **FINAL, "seed": SEED,
     "macro_f1_validasi": round(float(macro), 6),
     "feature_engineering": "tidak ada fitur turunan yang diadopsi (Fase 3)"},
    ensure_ascii=False, indent=2))
print("tersimpan ke", MODEL_DIR)

In [ ]:
# Muat ulang di kondisi bersih (simulasi sesi baru) dan pastikan prediksi identik.
loaded = np.load(MODEL_DIR / "knn_model.npz")
pre2 = KNNPreprocessor.from_dict(json.loads((MODEL_DIR / "preprocessing.json").read_text()))
cfg = json.loads((MODEL_DIR / "konfigurasi.json").read_text())
model2 = KNNClassifier(n_neighbors=cfg["k"], weights=cfg["weights"],
                       metric=cfg["metric"], threshold=cfg["threshold"]).fit(loaded["X_train"], loaded["y_train"])
pred2 = model2.predict(pre2.transform(test.drop(columns=["id"])), batch_size=BATCH)
np.testing.assert_array_equal(pred2, submission["is_canceled"].to_numpy())
print("prediksi hasil muat-ulang identik dengan submission")